# Integracao: LoadBalancer + Logger + Metrics

Este notebook demonstra uma execucao simples da simulacao, integrando:
- `LoadBalancer` para roteamento
- `MetricsCollector` para eventos e metricas
- `SimulationLogger` para logs estruturados

In [ ]:
import io
import logging
import os
import sys

import simpy

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'lab1':
    ROOT = os.path.abspath(os.getcwd())

SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from load_balancer_sim.config import SimulationConfig
from load_balancer_sim.load_balancer import LoadBalancer
from load_balancer_sim.logs import LOGGER_NAME, SimulationLogger
from load_balancer_sim.metrics import MetricsCollector
from load_balancer_sim.request import Request
from load_balancer_sim.server import Server

In [ ]:
# Configura logger para capturar saida em memoria e exibir no notebook.
log_buffer = io.StringIO()
stream_handler = logging.StreamHandler(log_buffer)
stream_handler.setFormatter(logging.Formatter('%(levelname)s %(message)s'))

logger = logging.getLogger(LOGGER_NAME)
logger.handlers.clear()
logger.setLevel(logging.DEBUG)
logger.addHandler(stream_handler)
logger.propagate = False

sim_logger = SimulationLogger(logger=logger)

In [ ]:
# Instanciacao do cenario da simulacao.
config = SimulationConfig(
    policy='round_robin',
    server_count=3,
    server_capacity=1,
    service_rate=20.0,
    horizon=1.0,
    warmup=0.0,
    seed=7,
)

environment = simpy.Environment()
collector = MetricsCollector(environment, event_handler=sim_logger.log_event)
servers = [
    Server(
        environment,
        server_id=index,
        capacity=config.server_capacity,
        service_rate=config.service_rate,
        seed=config.seed + index,
        on_service_started=collector.record_service_started,
        on_service_completed=collector.record_service_completed,
    )
    for index in range(config.server_count)
]

load_balancer = LoadBalancer(
    environment=environment,
    servers=servers,
    policy=config.policy,
    seed=config.seed,
)

sim_logger.log_run_started(config)

In [ ]:
def dispatch_request(env, lb, metrics_collector, request_id, burst_id=0):
    request = Request(id=request_id, burst_id=burst_id, arrival_time=env.now)
    metrics_collector.record_arrival(request)

    server = lb.route_request(request)
    metrics_collector.record_routing(request, server)
    return request, server

In [ ]:
# Envia algumas requisicoes quase simultaneas para observar roteamento e filas.
requests = []
for req_id in range(8):
    request, server = dispatch_request(environment, load_balancer, collector, req_id)
    requests.append((request, server.id))

# Executa a simulacao ate drenar os eventos agendados.
environment.run()

run_metrics = collector.calculate_run_metrics(
    horizon=config.horizon,
    warmup=config.warmup,
    servers=servers,
)
sim_logger.log_run_completed(run_metrics)

In [ ]:
print('Resumo de metricas:')
print(run_metrics)

print('\nPrimeiros 12 eventos metricos:')
for event in collector.events[:12]:
    print(event)

print('\nEstado final dos servidores:')
for server in servers:
    print({
        'server_id': server.id,
        'completed_count': server.completed_count,
        'max_active': server.maximum_active_count,
        'max_waiting': server.maximum_waiting_count,
    })

In [ ]:
print('Saida de logs estruturados:')
print(log_buffer.getvalue())